In [ ]:
import geemap
import ee 
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
geemap.ee_initialize()

In [ ]:
Map = geemap.Map()

In [ ]:
# Define the Relative Path
# '..' means go up one folder from where this notebook is saved
file_path = os.path.join('..', 'data', 'processed', 'Dholera_Taluk.geojson')

In [ ]:
roi = geemap.geojson_to_ee(file_path)
Map.addLayer(roi, {'color': 'blue', 'width': 2}, 'Dholera Taluka Boundary')

In [ ]:
# Load and Filter Sentinel-2 Surface Reflectance (2026)
# We use the Harmonized collection for consistency
s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(roi) \
    .filterDate('2025-10-01', '2025-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(roi)

In [ ]:
#Visualising this: 
vis = {
    "min": 0.0,
    "max": 3000,
    "bands": ["B4", "B3", "B2"],
}
Map.centerObject(roi, 10)
Map.addLayer(s2, vis, "Sentinel-2")
Map


Program to Check NDBI 

In the red/orange areas, you are likely looking at residential or commercial zones.

In [ ]:
# NDBI: Built-up/Urban Index
def get_ndbi(image):
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return ndbi

# --- Map 1: NDBI (Built-up / Urban) ---
Map_NDBI = geemap.Map()
Map_NDBI.centerObject(roi, 10)
ndbi_params = {'min': -0.5, 'max': 0.5, 'palette': ['white', 'orange', 'red']}

Map_NDBI.addLayer(get_ndbi(s2), ndbi_params, 'NDBI (Urban)')
Map_NDBI.add_colorbar(vis_params=ndbi_params, label="NDBI Value", orientation="horizontal")
print("Displaying NDBI Map...")
display(Map_NDBI)

Program to check MNDWI 

Look for the deep blue pixels; those will be your lakes, rivers, or reservoirs.

In [ ]:
# MNDWI: Water Index
def get_mndwi(image):
    mndwi = image.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    return mndwi

# --- Map 2: MNDWI (Water Index) ---
Map_MNDWI = geemap.Map()
Map_MNDWI.centerObject(roi, 10)
mndwi_params = {'min': -0.5, 'max': 0.5, 'palette': ['white', 'lightblue', 'blue']}

Map_MNDWI.addLayer(get_mndwi(s2), mndwi_params, 'MNDWI (Water)')
Map_MNDWI.add_colorbar(vis_params=mndwi_params, label="MNDWI Value", orientation="horizontal")
print("Displaying MNDWI Map...")
display(Map_MNDWI)

Program to check SAVI

The vibrant green represents healthy biomass. If you compare this to a standard NDVI, you'll notice it handles the edges of fields (where soil is visible) much more accurately.

In [ ]:
# SAVI: Soil Adjusted Vegetation Index
def get_savi(image):
    savi = image.expression(
        '((NIR - RED) * 1.5) / (NIR + RED + 0.5)', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')
    return savi

# --- Map 3: SAVI (Soil Adjusted Vegetation) ---
Map_SAVI = geemap.Map()
Map_SAVI.centerObject(roi, 10)
savi_params = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}

Map_SAVI.addLayer(get_savi(s2), savi_params, 'SAVI (Vegetation)')
Map_SAVI.add_colorbar(vis_params=savi_params, label="SAVI Value", orientation="horizontal")
print("Displaying SAVI Map...")
display(Map_SAVI)

Defining Indexes for Build-up Estimation

In [ ]:
# 2. Define Index Functions
def calculate_indices(image):
    # Band aliases for Sentinel-2
    # B11=SWIR1, B8=NIR, B3=Green, B4=Red
    res = image.expression(
        '(SWIR1 - NIR) / (SWIR1 + NIR)', {
            'SWIR1': image.select('B11'),
            'NIR': image.select('B8')
        }).rename('NDBI')
    
    mndwi = image.expression(
        '(Green - SWIR1) / (Green + SWIR1)', {
            'Green': image.select('B3'),
            'SWIR1': image.select('B11')
        }).rename('MNDWI')
    
    savi = image.expression(
        '((NIR - Red) * 1.5) / (NIR + Red + 0.5)', {
            'NIR': image.select('B8'),
            'Red': image.select('B4')
        }).rename('SAVI')
        
    return image.addBands([res, mndwi, savi])

# Apply the indices
processed_img = calculate_indices(s2)

Program to check and fine-tune the logic of Urban Build-up measurement

In [ ]:
# 3. Create the "Clean" Built-up Mask (The Logic)
# Thresholds: 
# NDBI > 0 (Built-up typically positive)
# MNDWI < 0 (Exclude water/moist salt pans)
# SAVI < 0.2 (Exclude vegetation)
built_up_mask = processed_img.expression(
    '(NDBI > 0.05) && (MNDWI < 0) && (SAVI < 0.18)', {
        'NDBI': processed_img.select('NDBI'),
        'MNDWI': processed_img.select('MNDWI'),
        'SAVI': processed_img.select('SAVI')
    }).rename('BuiltUp')

In [ ]:
# 4. Visualization
Map = geemap.Map()
Map.centerObject(roi, 11)

# True Color for Reference
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2',], 'min': 0, 'max': 3000}, 'S2 True Color (2025)')

# The Result: 1 = White (Built-up), 0 = Black (Everything Else)
Map.addLayer(built_up_mask, {'min': 0, 'max': 1, 'palette': ['black', 'white']}, 'Clean Built-up Layer')

Map

Exporting Image

In [ ]:
# Export the 2025 Built-up Mask to Google Drive
task = ee.batch.Export.image.toDrive(
    image= built_up_mask.toByte(), # Convert to 0-1 byte for smaller file size
    description='Dholera_Builtup_Mask_2025',
    folder='export',
    fileNamePrefix='dholera_builtup_2025',
    region=roi.geometry(),
    scale=10, # Sentinel-2 resolution
    crs='EPSG:32643', # Export in UTM 43N for direct QGIS use
    maxPixels=1e13
)
task.start()
print(" Export started! Check your Google Drive 'Dholera_Project' folder in a few minutes.")

I am providing the geojson file for important_roads, you can access that in data/processed folder. 
But, if you want latest road data, you can use QGIS OSM Pluggin to extract roads: In highway (motorway|trunk|primary|secondary|tertiary) roads are extracted. These are the most important roads in our analysis ensuring we utlise the most important roads only. 

In [ ]:
# 1: Local Infrastructure Ingestion ---
# Path to your manually verified roads from QGIS
roads_path = os.path.join('..', 'data', 'Processed', 'important_roads.geojson')

if os.path.exists(roads_path):
    print(f" Loading verified infrastructure from: {roads_path}")
    
    # 1. Load the local GeoJSON into a GeoDataFrame
    major_roads_gdf = gpd.read_file(roads_path)
    
    # Ensure it is in EPSG:4326 before sending to GEE
    if major_roads_gdf.crs != "EPSG:4326":
        major_roads_gdf = major_roads_gdf.to_crs("EPSG:4326")
    
    # 2. Upload to GEE Cloud
    ee_roads = geemap.geopandas_to_ee(major_roads_gdf)
    print(f" {len(major_roads_gdf)} road segments uploaded to GEE.")
else:
    print(" ERROR: File not found. Please check the path.")

In [ ]:
#2. Visualize Uploaded Infrastructure ---

# Create the Map
Map_Roads = geemap.Map()

# Define visualization parameters for the lines
road_vis = {
    'color': 'FF0000', # Bright Red
    'width': 2         # Thickness of the line
}

# Add the layer and center the map on the roads
Map_Roads.addLayer(ee_roads, road_vis, 'Verified Important Roads')
Map_Roads.centerObject(ee_roads, zoom=12)

# Display the map
Map_Roads

In [ ]:
# 3: GEE Raster Preparation ---

# Distance Raster (Threshold at 20km)
# The distance is calculated in the cloud based on the geometry of ee_roads
dist_raster = ee_roads.distance(20000).rename('dist_m')

# Built-up Density (Neighborhood scale 250m)
# .toFloat() ensures we get decimal density gradients (0.0 to 1.0)
density_raster = built_up_mask.toFloat().focal_mean(radius=250, units='meters').rename('builtup_density')

# Stack Bands into a single 'Analysis Image'
data_stack = dist_raster.addBands(density_raster)

In [ ]:
# 4: Sampling & Export ---

# 1. Generate 2000 Random "Survey" Points inside the ROI
sample_points = ee.FeatureCollection.randomPoints(region=roi, points=2000, seed=42)

# 2. ReduceRegions: The "Engine Room"
# We specify the CRS here to ensure 'dist_m' is interpreted as meters
sampled_fc = data_stack.reduceRegions(
    collection=sample_points,
    reducer=ee.Reducer.first(),
    scale=10,
    crs='EPSG:32643' #Based on Dholera location
)

Code section below will take time, because it is loading those 2000 points data in local memory. 
You can ideally skip it if you don't want to wait much, as I have provided csv data separately in data folder. 
Just run the code below this code block.

In [ ]:
print(" Extracting data and coordinates...")

# Get the raw data (which includes geometry)
raw_data = sampled_fc.getInfo()['features']

# Extract properties AND coordinates manually
data_list = []
for f in raw_data:
    props = f['properties']
    # Grab coordinates from the geometry key
    props['longitude'] = f['geometry']['coordinates'][0]
    props['latitude'] = f['geometry']['coordinates'][1]
    data_list.append(props)

# Create DataFrame
df_2025 = pd.DataFrame(data_list)

In [ ]:
# Final Cleaning
# Remove points that might have landed in 'No Data' zones
df_2025 = df_2025.dropna(subset=['dist_m', 'builtup_density'])

print(f"Success! Final dataset contains {len(df_2025)} points.")
print(df_2025.head())

This is a pure code just for the purpose to export to the drive.

In [ ]:
# --- STEP: ADD COORDINATES & EXPORT ---

# 1. Function to pull Lat/Lon out of the geometry and into properties (columns)
def add_lat_lon(feature):
    coords = feature.geometry().coordinates()
    return feature.set({
        'longitude': coords.get(0),
        'latitude': coords.get(1)
    })

# 2. Map the function over your collection
final_export_fc = sampled_fc.map(add_lat_lon)

# 3. Start the Export Task
task = ee.batch.Export.table.toDrive(
    collection=final_export_fc,
    description='Dholera_Sampling_2025_Final',
    folder='export',
    fileNamePrefix='dholera_samples_with_coords',
    fileFormat='CSV',
    selectors=['latitude', 'longitude', 'dist_m', 'builtup_density'] # Define column order
)

task.start()

print("Export Task started!")

# If you want to see the same thing, but wanna do it quickly, just run the code below.

In [ ]:
csv_path = os.path.join('..', 'data', 'processed', 'dholera_points_2025.csv')

if os.path.exists(csv_path):
    print("Loading pre-computed sampling data for instant analysis...")
    df_2025 = pd.read_csv(csv_path)
    
    # Quick cleanup to ensure data types are correct
    df_2025 = df_2025.dropna(subset=['dist_m', 'builtup_density'])
    print(f"Success! {len(df_2025)} points loaded instantly.")
else:
    print(" Error: CSV not found. Please run the GEE sampling cell first.")

df_2025.head()

The codes to map below are for verification purpose to check if the code is working fine or not.

In [ ]:
# Map 1: All Points
center_lat, center_lon = df_2025['latitude'].mean(), df_2025['longitude'].mean()
Map1 = geemap.Map(center=[center_lat, center_lon], zoom=11)
ee_all = geemap.pandas_to_ee(df_2025, latitude='latitude', longitude='longitude')
Map1.addLayer(ee_all, {'color': 'blue'}, 'All Points')
Map1

In [ ]:
# Map 2: Nearest 100
nearest_100 = df_2025.sort_values(by='dist_m').head(100)
Map2 = geemap.Map(center=[center_lat, center_lon], zoom=11)
ee_near = geemap.pandas_to_ee(nearest_100, latitude='latitude', longitude='longitude')
Map2.addLayer(ee_near, {'color': 'red'}, 'Nearest 100 to Roads')

print("Done! Map 1 (All) and Map 2 (Nearest 100) are ready.")
Map2 # Display the nearest 100 map

In the code below, we are trying to visualize the maximum build_up points over the actual build_up sentinel layer we created using Sentinel_2 data 

In [ ]:
# 1. Sort by builtup_density (descending = True for highest first)
top_density_df = df_2025.sort_values(by='builtup_density', ascending=False).head(200)

# 2. Convert this subset to a GEE object
ee_hotspots = geemap.pandas_to_ee(top_density_df, latitude='latitude', longitude='longitude')

# 3. Create the Map
Map = geemap.Map()
Map.centerObject(roi, 11)

# --- Layer 1: True Color Satellite Imagery ---
Map.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'S2 True Color (2025)')

# --- Layer 2: The Binary Mask (Semi-transparent) ---
# We use opacity=0.6 so we can see the satellite imagery underneath
Map.addLayer(built_up_mask, {'min': 0, 'max': 1, 'palette': ['black', 'white']}, 'Clean Built-up Layer', opacity=0.6)

# --- Layer 3: The Hotspots (Top 100 Density) ---
# Using a bright color like 'yellow' or 'cyan' to pop against the white mask
Map.addLayer(ee_hotspots, {'color': 'cyan'}, 'Top 100 Urban Hotspots')

# 4. Print the highest density found to check the data
print(f"Max density found: {top_density_df['builtup_density'].max():.4f}")

Map

Now we are finally entering in the domain to create Scatter Plot

In [ ]:
print(df_2025['dist_m'].describe())

In [ ]:

# 1. Start with the clean data
# We use .copy() to avoid the "SettingWithCopy" warning in Pandas
final_analysis_df = df_2025.dropna(subset=['dist_m', 'builtup_density']).copy()

# 2. Apply the "Dholera Infrastructure Buffer" (7.5 km)
# This includes the airport but excludes the distant rural outliers
buffer_limit = 7500 
impact_zone_df = final_analysis_df[final_analysis_df['dist_m'] <= buffer_limit].copy()

# 3. Handle zero-distance for logarithmic scaling (optional but recommended)
impact_zone_df['dist_m'] = impact_zone_df['dist_m'].replace(0, 1)

# 4. Summary of the final dataset
excluded_count = len(final_analysis_df) - len(impact_zone_df)
print(f"✅ Final Dataset Ready!")
print(f"---")
print(f"Points within 7.5km: {len(impact_zone_df)}")
print(f"Points excluded:      {excluded_count}")
print(f"Max distance kept:   {impact_zone_df['dist_m'].max():.2f} meters")
print(f"Average Density:     {impact_zone_df['builtup_density'].mean():.4f}")

# Preview the most distant points (to see your airport-area data)
print("\n--- Top 5 Distant Points (Airport Zone) ---")
print(impact_zone_df.sort_values('dist_m', ascending=False).head())

In [ ]:
# --- FINAL UNIFIED PLOTTING & SAVING CELL ---
import os

# 0. Set the visual style and initialize figure
sns.set_theme(style="whitegrid") 
plt.figure(figsize=(14, 8))

# 1. Create the plot
# Using order=2 to capture the non-linear "decay" of urban density
plot = sns.regplot(
    data=impact_zone_df, 
    x='dist_m', 
    y='builtup_density',
    scatter_kws={'alpha': 0.15, 's': 20, 'color': '#34495e', 'edgecolor': 'none'}, 
    line_kws={'color': '#e74c3c', 'lw': 4, 'label': 'Urban Decay Trend (Poly)'},
    order=2, 
    truncate=True
)

# 2. Strategic Annotations
# Primary Zone (<2km)
plt.axvline(2000, color='#27ae60', linestyle='--', lw=2, alpha=0.7)
plt.text(2100, impact_zone_df['builtup_density'].max()*0.9, 'Core Development Zone', color='#27ae60', fontweight='bold')

# Airport/Infrastructure Zone (~7km)
plt.axvline(7000, color='#2980b9', linestyle=':', lw=2, alpha=0.7)
plt.text(7100, impact_zone_df['builtup_density'].max()*0.8, 'Infrastructure Hub (Airport)', color='#2980b9', fontweight='bold')

# 3. Dynamic Y-Axis Scaling
plt.ylim(-0.01, impact_zone_df['builtup_density'].max() * 1.1)
plt.xlim(0, 7600)

# 4. Customizing Labels
plt.title('Dholera SIR: Urban Activation Signal\nDistance from Industrial Spine vs. Built-up Density', 
          fontsize=18, fontweight='bold', pad=25, loc='left', color='#2c3e50')
plt.xlabel('Distance to Main Road Network (Meters)', fontsize=13, labelpad=10)
plt.ylabel('Calculated Built-up Density (Normalized)', fontsize=13, labelpad=10)

plt.legend(loc='upper right', frameon=True, shadow=True)
sns.despine() 
plt.tight_layout()

# --- 5. THE SAVE STEP (Must be before plt.show()) ---
output_dir = os.path.join('..', 'output')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

save_path = os.path.join(output_dir, 'dholera_spatial_analysis_2025.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')

print(f"✅ Plot successfully saved to: {save_path}")

# 6. Final Display
plt.show()

In [ ]:
# --- 1. SETUP & STYLE ---
sns.set_theme(style="whitegrid") 
plt.figure(figsize=(14, 8))

# --- 2. THE PLOT ---
# order=2 allows the "U-shape" polynomial trend to show road AND airport influence
plot = sns.regplot(
    data=impact_zone_df, 
    x='dist_m', 
    y='builtup_density',
    scatter_kws={'alpha': 0.15, 's': 20, 'color': '#34495e', 'edgecolor': 'none'}, 
    line_kws={'color': '#e74c3c', 'lw': 4, 'label': 'Urban Activation Trend (Poly)'},
    order=2, 
    truncate=True
)

# --- 3. THE "STORYTELLING" ANNOTATIONS ---
# Primary Road Impact Zone
plt.axvline(2000, color='#27ae60', linestyle='--', lw=2, alpha=0.7)
plt.text(2100, 0.58, 'Core Development Zone', color='#27ae60', fontweight='bold')

# Airport Impact Zone
plt.axvline(7000, color='#2980b9', linestyle=':', lw=2, alpha=0.7)
plt.text(7100, 0.53, 'Infrastructure Hub (Airport)', color='#2980b9', fontweight='bold')

# --- 4. AXIS & LABELS (THE ZOOM UPDATE) ---
plt.ylim(-0.01, 0.65) 
plt.xlim(0, 7600)

plt.title('Dholera SIR: Dual-Anchor Urban Activation\nRoad Infrastructure vs. Airport Proximity', 
          fontsize=18, fontweight='bold', pad=25, loc='left', color='#2c3e50')
plt.xlabel('Distance to Main Road Network (Meters)', fontsize=13, labelpad=10)
plt.ylabel('Calculated Built-up Density (Normalized)', fontsize=13, labelpad=10)

plt.legend(loc='upper right', frameon=True, shadow=True)
sns.despine() 
plt.tight_layout()

# --- 5. THE SAVE STEP ---
output_dir = os.path.join('..', 'output')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

save_path = os.path.join(output_dir, 'dholera_spatial_analysis_2025(2).png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')

print(f"✅ Enhanced Plot saved to: {save_path}")

# --- 6. DISPLAY ---
plt.show()

Correlation Coefficient ($r$)

While $R^2$ tells you "How much," the Correlation Coefficient ($r$) tells you "Which Direction" and "How Strong." It ranges from -1 to +1.

Positive ($+1$): As Distance increases, Density increases (Unlikely for cities).

Negative ($-1$): As Distance increases, Density decreases (The "Standard" Urban Decay).

Near Zero ($0$): No linear relationship.

In [ ]:
# --- CALCULATE PEARSON CORRELATION ---
correlation = impact_zone_df['dist_m'].corr(impact_zone_df['builtup_density'])

print(f"📊 LINEAR ANALYSIS (Pearson's r)")
print(f"--------------------------------")
print(f"Correlation Coefficient: {correlation:.4f}")

if correlation < -0.5:
    print("Finding: Strong Negative Relationship. Infrastructure strongly dictates density.")
elif -0.5 <= correlation < -0.2:
    print("Finding: Moderate Negative Relationship. Typical of emerging urban zones.")
else:
    print("Finding: Weak Relationship. Other factors (zoning, land cost) are likely more dominant.")

Code to calculate R-squared 

$R^2 = 1.0$ (Perfect): Distance explains everything. Every building is exactly where the road tells it to be.

$R^2 = 0.0$ (Random): Distance explains nothing. Buildings are scattered like spilled salt.

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

# 1. Fit the 2nd order polynomial (y = ax^2 + bx + c)
# This is the mathematical 'brain' behind the red line in your plot
weights = np.polyfit(impact_zone_df['dist_m'], impact_zone_df['builtup_density'], 2)
model = np.poly1d(weights)

# 2. Generate predictions based on the model
y_true = impact_zone_df['builtup_density']
y_pred = model(impact_zone_df['dist_m'])

# 3. Calculate R-squared
poly_r2 = r2_score(y_true, y_pred)

print(f"📈 COMPLEX SPATIAL ANALYSIS (2nd Order Polynomial)")
print(f"--------------------------------------------------")
print(f"Polynomial R-squared: {poly_r2:.4f}")
print(f"Interpretation: {poly_r2*100:.1f}% of the variance in built-up density")
print(f"is explained by the Dual-Anchor model (Road Spine + Airport Hub).")

# Extra: The 'Direction' of the curve (The 'a' in ax^2 + bx + c)
if weights[0] > 0:
    print("\nStatistical Note: The positive quadratic term confirms a 'U-shaped' recovery,")
    print("statistically validating the impact of the distal Infrastructure Hub (Airport).")